# Gold Compliance Engine

This notebook applies metadata governance rules to validated Silver metadata, generates compliance scores, assigns certification status, and produces governance metrics for Genie AI and stakeholder dashboards.

### Governance Rules

- R01 - Table & Column Description Completeness
- R02 - Data Steward Assignment
- R03 - Security Classification Coverage
- R04 - Structural Compliance
- R05 - Critical Data Element Governance
- R06 - Source System Identification
- R07 - PII Classification Coverage
- R08 - Metadata Coverage Score



# STEP 0 — Promote Silver to Gold

Move validated metadata from the Silver layer to the Gold layer. This creates the trusted source used for governance, dashboards, and Genie AI analysis.

In [0]:
from pyspark.sql.functions import (
    col, when, lit, concat_ws, length, trim,
    avg, count, sum as spark_sum, current_timestamp,
    lower, countDistinct
)

In [0]:
silver_df = spark.table("metadata_governance.silver.silver_metadata")
silver_count = silver_df.count()

spark.sql("DROP TABLE IF EXISTS metadata_governance.gold.gold_metadata")

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("metadata_governance.gold.gold_metadata")

gold_check = spark.table("metadata_governance.gold.gold_metadata").count()

assert silver_count == gold_check, \
    f"Row count mismatch! Silver={silver_count} Gold={gold_check}"

print("=" * 70)
print("STEP 0 — SILVER PROMOTED TO GOLD")
print("=" * 70)
print(f"Silver rows read  : {silver_count}")
print(f"Gold rows written : {gold_check}")
print("=" * 70)


# STEP 1 — Load Gold Metadata

Load certified metadata from the Gold layer. This dataset will be evaluated against governance rules and used for compliance reporting.

In [0]:
gold_df = spark.table("metadata_governance.gold.gold_metadata")
gold_count = gold_df.count()

print("=" * 70)
print("STEP 1 — GOLD LAYER LOADED")
print("=" * 70)
print(f"Gold rows loaded  : {gold_count}")
print(f"Columns available : {len(gold_df.columns)}")
print("=" * 70)

display(gold_df.limit(5))


# STEP 2 — Load Quarantine Records

Load metadata records that failed previous validation checks. These records will be included in governance reporting to provide complete visibility.

In [0]:
quarantine_df = spark.table("metadata_governance.silver.bronze_metadata_quarantine")
quarantine_count = quarantine_df.count()

expected_total = 10000
actual_total = gold_count + quarantine_count

assert actual_total == expected_total, \
    f"Total count mismatch! Expected {expected_total}, got {actual_total}"

print("=" * 70)
print("STEP 2 — QUARANTINE RECORDS LOADED")
print("=" * 70)
print(f"Gold rows       : {gold_count}")
print(f"Quarantine rows : {quarantine_count}")
print(f"Total records   : {actual_total}")
print("=" * 70)

display(quarantine_df.limit(5))


# STEP 3 — Define Governance Rules

Define the governance framework consisting of 8 rules covering metadata quality, ownership, security, structure, and compliance readiness.

In [0]:
valid_security = ["public", "internal", "confidential", "restricted"]

expected_columns_per_table = 10
minimum_required_columns = int(expected_columns_per_table * 0.8)

invalid_source_values = [
    "unknown", "n/a", "na", "none", "null", "other", "test", "sample"
]


def ensure_column(df, target_col, possible_names):
    existing = df.columns

    if target_col in existing:
        return df

    for name in possible_names:
        if name in existing:
            return df.withColumn(target_col, col(name))

    return df.withColumn(target_col, lit(None))


def prepare_metadata_df(df):
    df = ensure_column(df, "source_system", [
        "source_system", "source_system_name", "system_name", "system", "tag_value"
    ])

    required_base_cols = [
        "database_name", "schema_name", "table_name", "column_name",
        "column_desc", "table_desc", "data_steward",
        "security_classification", "source_system",
        "pii_flag", "critical_data_element_flag"
    ]

    for c in required_base_cols:
        if c not in df.columns:
            df = df.withColumn(c, lit(None))

    if "actual_column_count" in df.columns:
        df = df.drop("actual_column_count")

    table_counts = df.groupBy(
        "database_name", "schema_name", "table_name"
    ).agg(
        countDistinct("column_name").alias("actual_column_count")
    )

    df = df.join(
        table_counts,
        on=["database_name", "schema_name", "table_name"],
        how="left"
    )

    return df


gold_df = prepare_metadata_df(gold_df)
quarantine_df = prepare_metadata_df(quarantine_df)


# R01 — Table & Column Description Completeness
r01_description = (
    col("column_desc").isNotNull() &
    (length(trim(col("column_desc"))) >= 5) &
    col("table_desc").isNotNull() &
    (length(trim(col("table_desc"))) >= 5)
)


# R02 — Data Steward Assignment
r02_steward = (
    col("data_steward").isNotNull() &
    (trim(col("data_steward")) != "")
)


# R03 — Security Classification Coverage
r03_security = (
    col("security_classification").isNotNull() &
    lower(trim(col("security_classification"))).isin(valid_security)
)


# R04 — Structural Compliance
r04_structure = (
    col("actual_column_count").isNotNull() &
    (col("actual_column_count") >= minimum_required_columns)
)


# R05 — Critical Data Element Governance
# Non-critical columns pass automatically.
# Critical columns must have description, steward, and classification.
r05_cde_governance = (
    (col("critical_data_element_flag") != True) |
    (
        (col("critical_data_element_flag") == True) &
        col("column_desc").isNotNull() &
        (length(trim(col("column_desc"))) >= 10) &
        col("data_steward").isNotNull() &
        (trim(col("data_steward")) != "") &
        col("security_classification").isNotNull() &
        (trim(col("security_classification")) != "")
    )
)


# R06 — Source System Governance
# Source system must be populated, meaningful, and not a generic placeholder.
r06_source = (
    col("source_system").isNotNull() &
    (trim(col("source_system")) != "") &
    (~lower(trim(col("source_system"))).isin(invalid_source_values)) &
    (length(trim(col("source_system"))) >= 3)
)


# R07 — PII Governance Alignment
# PII flag must exist.
# If PII = true, security classification must be confidential or restricted.
r07_pii = (
    col("pii_flag").isNotNull() &
    (
        (col("pii_flag") != True) |
        (
            (col("pii_flag") == True) &
            lower(trim(col("security_classification"))).isin(["confidential", "restricted"])
        )
    )
)


# R08 — Metadata Coverage Score
# Pass if at least 5 out of 7 important metadata fields are populated.
r08_metadata_coverage = (
    (
        when(col("column_desc").isNotNull() & (trim(col("column_desc")) != ""), 1).otherwise(0) +
        when(col("table_desc").isNotNull() & (trim(col("table_desc")) != ""), 1).otherwise(0) +
        when(col("data_steward").isNotNull() & (trim(col("data_steward")) != ""), 1).otherwise(0) +
        when(col("security_classification").isNotNull() & (trim(col("security_classification")) != ""), 1).otherwise(0) +
        when(col("source_system").isNotNull() & (trim(col("source_system")) != ""), 1).otherwise(0) +
        when(col("pii_flag").isNotNull(), 1).otherwise(0) +
        when(col("critical_data_element_flag").isNotNull(), 1).otherwise(0)
    ) >= 5
)


print("=" * 70)
print("STEP 3 — GOVERNANCE RULES DEFINED R01-R08")
print("=" * 70)
print("R01 — Table & column descriptions populated")
print("R02 — Data steward assigned")
print("R03 — Security classification valid")
print("R04 — Structural compliance, at least 80% expected columns")
print("R05 — Critical data element governance")
print("R06 — Source system populated and valid")
print("R07 — PII classification aligned with security level")
print("R08 — At least 70% metadata coverage")
print("=" * 70)


# STEP 4 — Apply Governance Rules

Evaluate each metadata record against the governance rules, calculate compliance scores, and assign certification status.

In [0]:
def apply_governance_rules(df, ingestion_status_value):
    return df \
        .withColumn("r01_pass", when(r01_description, lit(1)).otherwise(lit(0))) \
        .withColumn("r02_pass", when(r02_steward, lit(1)).otherwise(lit(0))) \
        .withColumn("r03_pass", when(r03_security, lit(1)).otherwise(lit(0))) \
        .withColumn("r04_pass", when(r04_structure, lit(1)).otherwise(lit(0))) \
        .withColumn("r05_pass", when(r05_cde_governance, lit(1)).otherwise(lit(0))) \
        .withColumn("r06_pass", when(r06_source, lit(1)).otherwise(lit(0))) \
        .withColumn("r07_pass", when(r07_pii, lit(1)).otherwise(lit(0))) \
        .withColumn("r08_pass", when(r08_metadata_coverage, lit(1)).otherwise(lit(0))) \
        .withColumn(
            "completeness_score",
            (
                col("r01_pass") +
                col("r02_pass") +
                col("r03_pass") +
                col("r04_pass") +
                col("r05_pass") +
                col("r06_pass") +
                col("r07_pass") +
                col("r08_pass")
            ) * lit(100) / lit(8)
        ) \
        .withColumn(
            "certification_status",
            when(col("completeness_score") >= 75, lit("PASS"))
            .otherwise(lit("FAIL"))
        ) \
        .withColumn(
            "certification_level",
            when(col("completeness_score") >= 85, lit("Certified"))
            .when(col("completeness_score") >= 70, lit("Compliant"))
            .when(col("completeness_score") >= 50, lit("Needs Improvement"))
            .otherwise(lit("Non-Compliant"))
        ) \
        .withColumn(
            "error_detail",
            concat_ws(" | ",
                when(~r01_description, lit("R01: Table or column description missing")),
                when(~r02_steward, lit("R02: Data steward missing")),
                when(~r03_security, lit("R03: Security classification missing or invalid")),
                when(~r04_structure, lit("R04: Table does not meet structural standard")),
                when(~r05_cde_governance, lit("R05: Critical data element governance requirements not met")),
                when(~r06_source, lit("R06: Source system missing or invalid")),
                when(~r07_pii, lit("R07: PII governance alignment failed")),
                when(~r08_metadata_coverage, lit("R08: Metadata coverage below threshold"))
            )
        ) \
        .withColumn(
            "remediation_hint",
            concat_ws(" | ",
                when(~r01_description, lit("Add meaningful table and column descriptions")),
                when(~r02_steward, lit("Assign a responsible data steward")),
                when(~r03_security, lit("Set classification to public, internal, confidential, or restricted")),
                when(~r04_structure, lit("Review missing columns and align table with expected structure")),
                when(~r05_cde_governance, lit("Add description, steward, and classification for critical data elements")),
                when(~r06_source, lit("Populate a valid source system")),
                when(~r07_pii, lit("Align PII classification with security requirements")),
                when(~r08_metadata_coverage, lit("Populate more required metadata fields"))
            )
        ) \
        .withColumn(
            "error_detail",
            when(col("certification_status") == "PASS", lit(None))
            .otherwise(col("error_detail"))
        ) \
        .withColumn(
            "remediation_hint",
            when(col("certification_status") == "PASS", lit(None))
            .otherwise(col("remediation_hint"))
        ) \
        .withColumn("ingestion_status", lit(ingestion_status_value)) \
        .drop(
            "r01_pass", "r02_pass", "r03_pass", "r04_pass",
            "r05_pass", "r06_pass", "r07_pass", "r08_pass"
        )


df_gold_certified = apply_governance_rules(gold_df, "PASSED_SILVER")

pass_gold = df_gold_certified.filter(col("certification_status") == "PASS").count()
fail_gold = df_gold_certified.filter(col("certification_status") == "FAIL").count()
pass_rate = pass_gold / gold_count * 100 if gold_count > 0 else 0

print("=" * 70)
print("STEP 4 — GOLD RECORDS CERTIFICATION RESULTS")
print("=" * 70)
print(f"Total Gold records        : {gold_count}")
print(f"Certification threshold   : 75%")
print(f"PASS                      : {pass_gold}")
print(f"FAIL                      : {fail_gold}")
print(f"Pass rate                 : {pass_rate:.2f}%")
print("=" * 70)

display(
    df_gold_certified.select(
        "table_name",
        "column_name",
        "certification_status",
        "certification_level",
        "completeness_score",
        "error_detail",
        "ingestion_status"
    ).limit(10)
)


# STEP 5 — Evaluate Quarantine Records

Apply governance checks to quarantined records and classify them as non-compliant while capturing remediation details.

In [0]:
df_quarantine_certified = apply_governance_rules(quarantine_df, "FAILED_SILVER")

df_quarantine_certified = df_quarantine_certified \
    .withColumn("certification_status", lit("FAIL")) \
    .withColumn("certification_level", lit("Non-Compliant"))

if "failure_reason" in df_quarantine_certified.columns:
    df_quarantine_certified = df_quarantine_certified.withColumn(
        "error_detail",
        concat_ws(" | ",
            concat_ws(": ", lit("INGESTION FAIL"), col("failure_reason")),
            col("error_detail")
        )
    ).drop("failure_reason")

quarantine_pass_check = df_quarantine_certified.filter(
    col("certification_status") == "PASS"
).count()

assert quarantine_pass_check == 0, \
    f"Quarantine records incorrectly marked as PASS: {quarantine_pass_check}"

print("=" * 70)
print("STEP 5 — QUARANTINE RECORDS CERTIFICATION RESULTS")
print("=" * 70)
print(f"Total quarantine records : {quarantine_count}")
print("All quarantine records   : FAIL")
print("=" * 70)

display(
    df_quarantine_certified.select(
        "table_name",
        "column_name",
        "certification_status",
        "certification_level",
        "completeness_score",
        "error_detail",
        "ingestion_status"
    ).limit(10)
)


# STEP 6 — Build Compliance View

Combine certified and quarantined records into a single governance dataset for reporting and analysis.

In [0]:
df_full = df_gold_certified.unionByName(
    df_quarantine_certified,
    allowMissingColumns=True
)

total_full = df_full.count()
pass_full = df_full.filter(col("certification_status") == "PASS").count()
fail_full = df_full.filter(col("certification_status") == "FAIL").count()

# Pass rate = how many records passed
overall_pass_rate = pass_full / total_full * 100 if total_full > 0 else 0

# Compliance score = average completeness score
overall_score = df_full.agg(
    avg(col("completeness_score"))
).collect()[0][0]

overall_score = safe_num(overall_score)

assert total_full == gold_count + quarantine_count, \
    f"Union row count mismatch! Expected {gold_count + quarantine_count}, got {total_full}"

print("=" * 70)
print("STEP 6 — COMPLETE COMPLIANCE PICTURE")
print("=" * 70)
print(f"Total records processed     : {total_full}")
print(f"Overall PASS                : {pass_full}")
print(f"Overall FAIL                : {fail_full}")
print(f"Certification pass rate     : {overall_pass_rate:.2f}%")
print(f"Overall compliance score    : {overall_score:.2f}%")
print("=" * 70)


# STEP 7 — Calculate Governance Metrics

Generate compliance KPIs, rule-level metrics, and governance insights to identify strengths and improvement areas.

In [0]:
def safe_num(value, default=0.0):
    return float(value) if value is not None else default


def status_label(score):
    if score >= 80:
        return "GOOD"
    elif score >= 50:
        return "NEEDS REVIEW"
    else:
        return "CRITICAL"


field_metrics = df_full.agg(
    avg(when(r01_description, lit(100)).otherwise(lit(0))).alias("R01_description_pct"),
    avg(when(r02_steward, lit(100)).otherwise(lit(0))).alias("R02_steward_pct"),
    avg(when(r03_security, lit(100)).otherwise(lit(0))).alias("R03_security_pct"),
    avg(when(r04_structure, lit(100)).otherwise(lit(0))).alias("R04_structure_pct"),
    avg(when(r05_cde_governance, lit(100)).otherwise(lit(0))).alias("R05_cde_governance_pct"),
    avg(when(r06_source, lit(100)).otherwise(lit(0))).alias("R06_source_pct"),
    avg(when(r07_pii, lit(100)).otherwise(lit(0))).alias("R07_pii_pct"),
    avg(when(r08_metadata_coverage, lit(100)).otherwise(lit(0))).alias("R08_metadata_coverage_pct")
)

field_row = field_metrics.collect()[0]

fields = [
    ("R01 Description Completeness", safe_num(field_row["R01_description_pct"])),
    ("R02 Data Steward Assignment", safe_num(field_row["R02_steward_pct"])),
    ("R03 Security Classification", safe_num(field_row["R03_security_pct"])),
    ("R04 Structural Compliance", safe_num(field_row["R04_structure_pct"])),
    ("R05 Critical Data Element Governance", safe_num(field_row["R05_cde_governance_pct"])),
    ("R06 Source System Governance", safe_num(field_row["R06_source_pct"])),
    ("R07 PII Governance Alignment", safe_num(field_row["R07_pii_pct"])),
    ("R08 Metadata Coverage", safe_num(field_row["R08_metadata_coverage_pct"]))
]

worst_field = min(fields, key=lambda x: x[1])
best_field = max(fields, key=lambda x: x[1])

print("=" * 70)
print("FIELD LEVEL GOVERNANCE METRICS")
print("=" * 70)

for name, score in fields:
    print(f"{status_label(score):<14} | {name:<45} | {score:>6.2f}%")

print("=" * 70)
print(f"Best performing rule : {best_field[0]} at {best_field[1]:.2f}%")
print(f"Biggest gap          : {worst_field[0]} at {worst_field[1]:.2f}%")
print("=" * 70)


rule_breakdown = df_full.agg(
    spark_sum(when(~r01_description, lit(1)).otherwise(lit(0))).alias("R01_failures"),
    spark_sum(when(~r02_steward, lit(1)).otherwise(lit(0))).alias("R02_failures"),
    spark_sum(when(~r03_security, lit(1)).otherwise(lit(0))).alias("R03_failures"),
    spark_sum(when(~r04_structure, lit(1)).otherwise(lit(0))).alias("R04_failures"),
    spark_sum(when(~r05_cde_governance, lit(1)).otherwise(lit(0))).alias("R05_failures"),
    spark_sum(when(~r06_source, lit(1)).otherwise(lit(0))).alias("R06_failures"),
    spark_sum(when(~r07_pii, lit(1)).otherwise(lit(0))).alias("R07_failures"),
    spark_sum(when(~r08_metadata_coverage, lit(1)).otherwise(lit(0))).alias("R08_failures")
)

bd_row = rule_breakdown.collect()[0]

rule_failures = [
    ("R01 Description Completeness", int(safe_num(bd_row["R01_failures"]))),
    ("R02 Data Steward Assignment", int(safe_num(bd_row["R02_failures"]))),
    ("R03 Security Classification", int(safe_num(bd_row["R03_failures"]))),
    ("R04 Structural Compliance", int(safe_num(bd_row["R04_failures"]))),
    ("R05 Critical Data Element Governance", int(safe_num(bd_row["R05_failures"]))),
    ("R06 Source System Governance", int(safe_num(bd_row["R06_failures"]))),
    ("R07 PII Governance Alignment", int(safe_num(bd_row["R07_failures"]))),
    ("R08 Metadata Coverage", int(safe_num(bd_row["R08_failures"])))
]

worst_rule = max(rule_failures, key=lambda x: x[1])

print("=" * 70)
print("RULE BREAKDOWN — FAILURE COUNT PER RULE")
print("=" * 70)

for name, failures in rule_failures:
    pct = failures / total_full * 100 if total_full > 0 else 0
    print(f"{name:<45} | {failures:>5} failures | {pct:>6.2f}%")

print("=" * 70)
print(f"Rule causing most failures: {worst_rule[0]} ({worst_rule[1]} failures)")
print("=" * 70)


table_metrics = df_full.groupBy("table_name").agg(
    count("*").alias("total_records"),
    spark_sum(when(col("certification_status") == "PASS", 1).otherwise(0)).alias("passed_records"),
    avg(col("completeness_score")).alias("table_compliance_pct")
).orderBy("table_compliance_pct")

schema_metrics = df_full.groupBy("schema_name").agg(
    count("*").alias("total_records"),
    spark_sum(when(col("certification_status") == "PASS", 1).otherwise(0)).alias("passed_records"),
    avg(col("completeness_score")).alias("schema_compliance_pct")
).orderBy("schema_compliance_pct")

db_metrics = df_full.groupBy("database_name").agg(
    count("*").alias("total_records"),
    spark_sum(when(col("certification_status") == "PASS", 1).otherwise(0)).alias("passed_records"),
    avg(col("completeness_score")).alias("database_compliance_pct")
).orderBy("database_compliance_pct")

overall_score = safe_num(overall_rate)

print("=" * 70)
print("OVERALL CATALOG COMPLIANCE SUMMARY")
print("=" * 70)
print(f"Overall compliance score : {overall_score:.2f}%")
print(f"Total PASS               : {pass_full}")
print(f"Total FAIL               : {fail_full}")
print(f"Total records            : {total_full}")
print(f"Biggest governance gap   : {worst_field[0]}")
print(f"Most rule failures       : {worst_rule[0]}")
print("=" * 70)


# STEP 8 — Publish Governance Results

Write certified metadata and governance metrics to the Gold layer for dashboards and Genie AI consumption.

In [0]:
from pyspark.sql import Row

certified_table_name = "metadata_governance.gold.gold_certified_metadata_final"
metrics_table_name = "metadata_governance.gold.gold_quality_metrics_final"

df_final = df_full.withColumn("certified_at", current_timestamp())

df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(certified_table_name)

certified_count = spark.table(certified_table_name).count()

assert certified_count == total_full, \
    f"Write count mismatch! Expected {total_full}, got {certified_count}"


def safe_float(value, default=0.0):
    return float(value) if value is not None else default


def safe_int(value, default=0):
    return int(value) if value is not None else default


metrics_rows = []

# Catalog metric
# compliance_score = average completeness_score
# pass rate is stored separately through passed/failed counts
metrics_rows.append(Row(
    level="catalog",
    name="metadata_governance",
    total_records=safe_int(total_full),
    passed=safe_int(pass_full),
    failed=safe_int(fail_full),
    compliance_score=safe_float(overall_score)
))

# Schema metrics
for row in schema_metrics.collect():
    metrics_rows.append(Row(
        level="schema",
        name=str(row["schema_name"]) if row["schema_name"] is not None else "Unknown schema",
        total_records=safe_int(row["total_records"]),
        passed=safe_int(row["passed_records"]),
        failed=safe_int(row["total_records"]) - safe_int(row["passed_records"]),
        compliance_score=safe_float(row["schema_compliance_pct"])
    ))

# Table metrics
for row in table_metrics.collect():
    metrics_rows.append(Row(
        level="table",
        name=str(row["table_name"]) if row["table_name"] is not None else "Unknown table",
        total_records=safe_int(row["total_records"]),
        passed=safe_int(row["passed_records"]),
        failed=safe_int(row["total_records"]) - safe_int(row["passed_records"]),
        compliance_score=safe_float(row["table_compliance_pct"])
    ))

# Database metrics
for row in db_metrics.collect():
    metrics_rows.append(Row(
        level="database",
        name=str(row["database_name"]) if row["database_name"] is not None else "Unknown database",
        total_records=safe_int(row["total_records"]),
        passed=safe_int(row["passed_records"]),
        failed=safe_int(row["total_records"]) - safe_int(row["passed_records"]),
        compliance_score=safe_float(row["database_compliance_pct"])
    ))

# Rule metrics
for rule_name, failures in rule_failures:
    failures = safe_int(failures)

    compliance = (
        (total_full - failures) / total_full * 100
        if total_full > 0 else 0
    )

    metrics_rows.append(Row(
        level="rule",
        name=str(rule_name),
        total_records=safe_int(total_full),
        passed=safe_int(total_full - failures),
        failed=safe_int(failures),
        compliance_score=safe_float(compliance)
    ))

# Field metrics
for field_name, field_score in fields:
    field_score = safe_float(field_score)

    field_fail = (
        int((1 - field_score / 100) * total_full)
        if total_full > 0 else 0
    )

    metrics_rows.append(Row(
        level="field",
        name=str(field_name),
        total_records=safe_int(total_full),
        passed=safe_int(total_full - field_fail),
        failed=safe_int(field_fail),
        compliance_score=safe_float(field_score)
    ))

df_metrics = spark.createDataFrame(metrics_rows) \
    .withColumn("calculated_at", current_timestamp())

df_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(metrics_table_name)

metrics_count = spark.table(metrics_table_name).count()

print("=" * 70)
print("STEP 8 — GOLD LAYER OUTPUT WRITTEN")
print("=" * 70)
print(f"Certified metadata table : {certified_table_name}")
print(f"Rows written             : {certified_count}")
print(f"Quality metrics table    : {metrics_table_name}")
print(f"Metrics rows written     : {metrics_count}")
print("=" * 70)


# STEP 9 — Final Validation

Validate the final outputs and generate a summary of compliance performance, governance gaps, and certification results.

In [0]:
gold_base = spark.table("metadata_governance.gold.gold_metadata")
gold_cert = spark.table("metadata_governance.gold.gold_certified_metadata_final")
gold_metrics = spark.table("metadata_governance.gold.gold_quality_metrics_final")

gold_count = gold_base.count()
certified_count = gold_cert.count()
metrics_count = gold_metrics.count()

pass_full = gold_cert.filter(col("certification_status") == "PASS").count()
fail_full = gold_cert.filter(col("certification_status") == "FAIL").count()

total_full = pass_full + fail_full

quarantine_count = gold_cert.filter(col("ingestion_status") == "FAILED_SILVER").count()

# Pass rate = number of PASS records / total records
overall_pass_rate = (pass_full / total_full * 100) if total_full > 0 else 0

# Compliance score = average completeness_score
overall_score = gold_cert.agg(
    avg(col("completeness_score"))
).collect()[0][0]

overall_score = safe_num(overall_score)

print("=" * 70)
print("GOLD LAYER VALIDATION SUMMARY")
print("=" * 70)
print(f"Gold base metadata table     : {gold_base.count()} rows")
print(f"Certified governance records : {gold_cert.count()} rows")
print(f"Quality metrics records      : {gold_metrics.count()} rows")
print("=" * 70)

print("\nSample PASS records")
display(
    gold_cert.filter(col("certification_status") == "PASS")
    .select(
        "table_name", "column_name",
        "certification_status", "certification_level",
        "completeness_score", "security_classification",
        "data_steward", "ingestion_status", "certified_at"
    )
    .limit(5)
)

print("\nSample FAIL records")
display(
    gold_cert.filter(col("certification_status") == "FAIL")
    .select(
        "table_name", "column_name",
        "certification_status", "certification_level",
        "completeness_score", "error_detail",
        "remediation_hint", "ingestion_status", "certified_at"
    )
    .limit(5)
)

print("\nGovernance metrics by rule")
display(
    gold_metrics.filter(col("level") == "rule")
    .orderBy("compliance_score")
)

print("\nGovernance metrics by table")
display(
    gold_metrics.filter(col("level") == "table")
    .orderBy("compliance_score")
)

print("\nCatalog governance overview")
display(
    gold_metrics.filter(col("level") == "catalog")
)

print("=" * 70)
print("FINAL GOVERNANCE PIPELINE SUMMARY")
print("=" * 70)
print("Original Bronze records processed : 10,000")
print(f"Records passed Silver ingestion   : {gold_count}")
print(f"Records sent to quarantine        : {quarantine_count}")
print("-" * 70)
print(f"Governance PASS records           : {pass_full}")
print(f"Governance FAIL records           : {fail_full}")
print(f"Certification pass rate           : {overall_pass_rate:.2f}%")
print(f"Overall compliance score          : {overall_score:.2f}%")
print("-" * 70)
print("Governance rules applied          : 8")
print("Gold layer tables created         : 3")
print(f"Quality metric records generated  : {metrics_count}")
print("=" * 70)

print("=" * 70)
print("GOLD COMPLIANCE ENGINE COMPLETED SUCCESSFULLY")
print("=" * 70)